# Glucose Prediction Preprocessing Workflow

This notebook builds a modeling-ready dataset from the HUPA Diabetes CSVs in `data/`.

**Pipeline**
1. Load all participant CSVs from `data/`
2. Fix timestamps
3. Handle missing values
4. Remove physiologically implausible outliers
5. Normalize features per participant
6. Merge all modalities into one tidy frame
7. Build sliding windows for glucose prediction
8. Persist modeling artifacts

## 1. Load Dataset from Data Folder

Each CSV in `data/` is one participant. Files are semicolon-delimited and share the same schema: `time, glucose, calories, heart_rate, steps, basal_rate, bolus_volume_delivered, carb_input`. We tag each row with a `participant_id` derived from the filename.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('data')
FEATURE_COLS = ['glucose', 'calories', 'heart_rate', 'steps',
                'basal_rate', 'bolus_volume_delivered', 'carb_input']

csv_files = sorted(DATA_DIR.glob('*.csv'))
frames = []
for path in csv_files:
    part = pd.read_csv(path, sep=';')
    part['participant_id'] = path.stem
    frames.append(part)

raw_df = pd.concat(frames, ignore_index=True)
print('Files loaded :', [p.name for p in csv_files])
print('Raw shape    :', raw_df.shape)
print('Per-participant rows:')
print(raw_df.groupby('participant_id').size())
raw_df.head()

## 2. Fix Timestamps

We parse `time` to `datetime64[ns]`, drop rows with unparseable timestamps, and floor each timestamp to the nearest 5-minute boundary so that downstream resampling lines up cleanly. Sorting by `(participant_id, time)` and dropping duplicate timestamps within a participant prevents resample ambiguity.

In [ ]:
df = raw_df.copy()
df['time'] = pd.to_datetime(df['time'], errors='coerce')

bad = df['time'].isna().sum()
print(f'Unparseable timestamps dropped: {bad}')
df = df.dropna(subset=['time'])

if df['time'].dt.tz is not None:
    df['time'] = df['time'].dt.tz_convert('UTC').dt.tz_localize(None)

df['time'] = df['time'].dt.floor('5min')
df = (df.sort_values(['participant_id', 'time'])
        .drop_duplicates(subset=['participant_id', 'time'], keep='first')
        .reset_index(drop=True))

print('Shape after timestamp fix:', df.shape)
print('Time range per participant:')
print(df.groupby('participant_id')['time'].agg(['min', 'max']))

## 3. Handle Missing Values

Sensor outages can leave NaN values in the raw rows. Strategy:

- **Continuous physiological signals** (`glucose`, `heart_rate`, `basal_rate`): time-aware interpolation, but only across short gaps (≤ 6 consecutive samples) so we don't fabricate long stretches of CGM data.
- **Event-like / count signals** (`calories`, `steps`, `bolus_volume_delivered`, `carb_input`): a missing value means "no event," so fill with 0.
- Drop any rows where `glucose` is still missing — we cannot train without a target.

In [ ]:
print('Missing values in raw data:')
print(df[FEATURE_COLS].isna().sum())

CONT = ['glucose', 'heart_rate', 'basal_rate']
EVENT = ['calories', 'steps', 'bolus_volume_delivered', 'carb_input']
MAX_GAP = 6  # 6 consecutive missing samples

imputed_parts = []
for pid, part in df.groupby('participant_id'):
    part = part.sort_values('time').set_index('time').copy()
    part[CONT] = part[CONT].interpolate(method='time', limit=MAX_GAP, limit_direction='both')
    part[EVENT] = part[EVENT].fillna(0)
    part = part.reset_index()
    part['participant_id'] = pid
    imputed_parts.append(part)

imputed = pd.concat(imputed_parts, ignore_index=True)

before = len(imputed)
imputed = imputed.dropna(subset=['glucose']).reset_index(drop=True)
print(f'\nDropped {before - len(imputed)} rows where glucose remained NaN after interpolation')
print('\nMissing values after imputation:')
print(imputed[FEATURE_COLS].isna().sum())
print('\nColumns:', imputed.columns.tolist())

## 4. Remove Outliers

We use **physiological plausibility bounds** rather than IQR. The dataset has very sparse insulin/carb columns (mostly zeros), so IQR rejects almost every nonzero event. Hard clinical bounds are safer:

- `glucose`: 40–400 mg/dL (CGM operating range)
- `heart_rate`: 30–220 bpm
- `steps`: 0–500 per 5 min (≈ 100 steps/min sprint cap)
- `calories`: 0–100 kcal per 5 min

Rows outside these bounds are dropped; insulin and carb columns are left untouched because their distribution is dominated by legitimate zeros and rare large events.

In [ ]:
BOUNDS = {
    'glucose':    (40, 400),
    'heart_rate': (30, 220),
    'steps':      (0, 500),
    'calories':   (0, 100),
}

before = len(imputed)
mask = pd.Series(True, index=imputed.index)
for col, (lo, hi) in BOUNDS.items():
    col_mask = imputed[col].between(lo, hi)
    print(f'{col:12s} kept {col_mask.sum():>5d} / {len(imputed)}  (range {lo}-{hi})')
    mask &= col_mask

cleaned = imputed[mask].reset_index(drop=True)
print(f'\nRows before: {before}, after: {len(cleaned)}, dropped: {before - len(cleaned)}')

## 5. Normalize Per Participant

Each participant has their own physiological baseline (resting HR, glucose set point, activity profile). Z-scoring within participant preserves intra-individual dynamics while removing inter-individual scale differences.

We keep the original `glucose` column as `glucose_raw` so the prediction target stays in mg/dL — only the model *inputs* are scaled.

In [ ]:
normalized = cleaned.copy()
normalized['glucose_raw'] = normalized['glucose']

scaled_parts = []
for pid, part in normalized.groupby('participant_id', group_keys=False):
    scaler = StandardScaler()
    part = part.copy()
    part[FEATURE_COLS] = scaler.fit_transform(part[FEATURE_COLS])
    scaled_parts.append(part)

normalized = pd.concat(scaled_parts, ignore_index=True)

print('Per-participant feature stats (should be ~0 mean / ~1 std):')
print(normalized.groupby('participant_id')[FEATURE_COLS].agg(['mean', 'std']).round(3).head())

## 6. Merge All Modalities

In the HUPA dataset, glucose (CGM), Fitbit (calories/HR/steps), and pump (basal/bolus/carbs) are already aligned in one file per participant and share a 5-minute timestamp. They merge by simple concatenation. We materialize the unified frame indexed by `(participant_id, time)`.

In [ ]:
merged = (normalized
          .set_index(['participant_id', 'time'])
          .sort_index())

print('Merged shape:', merged.shape)
print('Columns     :', merged.columns.tolist())
merged.head()

## 7. Create Modeling Windows for Glucose Prediction

We build sliding windows per participant:
- **History**: 12 windows × 5 min = **1 hour** of past features
- **Horizon**: 6 windows × 5 min = **30 min** ahead glucose target

A window is only emitted if the slice is **contiguous** (no time gap was bridged inside it) — otherwise we'd train on stitched-together segments. The target is `glucose_raw` (unscaled mg/dL) so model errors are interpretable.

In [ ]:
HISTORY_STEPS = 12   # 1 hour of history
HORIZON_STEPS = 6    # predict 30 min ahead
STEP_MINUTES = 5

def build_windows(df, feature_cols, target_col='glucose_raw',
                  history=HISTORY_STEPS, horizon=HORIZON_STEPS):
    X, y, meta = [], [], []
    expected_span = pd.Timedelta(minutes=STEP_MINUTES * (history + horizon - 1))

    for pid, part in df.groupby(level='participant_id'):
        part = part.sort_index()
        times = part.index.get_level_values('time')
        feats = part[feature_cols].to_numpy()
        target = part[target_col].to_numpy()
        n = len(part)
        last_start = n - history - horizon + 1

        for s in range(last_start):
            t_end = s + history + horizon - 1
            if times[t_end] - times[s] != expected_span:
                continue  # gap inside window; skip
            X.append(feats[s:s + history])
            y.append(target[t_end])
            meta.append({
                'participant_id': pid,
                'window_start': times[s],
                'target_time':  times[t_end],
            })
    return np.asarray(X), np.asarray(y), pd.DataFrame(meta)

X, y, meta = build_windows(merged, FEATURE_COLS)
print('X shape:', X.shape, '   # (n_windows, history_steps, n_features)')
print('y shape:', y.shape, '   # (n_windows,) glucose mg/dL at +30 min')
print('Windows per participant:')
print(meta.groupby('participant_id').size())
meta.head()

## 8. Persist the Modeling Artifacts (optional)

Save the windowed arrays and the merged tidy frame so downstream notebooks can load them directly. Numeric columns are rounded to 3 decimal places.

In [ ]:
out_dir = Path('processed')
out_dir.mkdir(exist_ok=True)

ROUND_COLS = ['glucose', 'heart_rate', 'basal_rate', 'calories', 'steps',
              'bolus_volume_delivered', 'carb_input', 'glucose_raw']

merged_out = merged.reset_index().copy()
merged_out[ROUND_COLS] = merged_out[ROUND_COLS].round(3)
merged_out.to_csv(out_dir / 'merged_5min.csv', index=False)

np.savez_compressed(out_dir / 'glucose_windows.npz',
                    X=np.round(X, 3), y=np.round(y, 3))
meta.to_csv(out_dir / 'glucose_windows_meta.csv', index=False)

print('Saved:')
for p in out_dir.iterdir():
    print(' ', p, f'({p.stat().st_size / 1024:.1f} KB)')